# E05 Credit Card Fraud：时间划分、极端不均衡与 PR-AUC

> Issue #9 / Epic #6 / 路线图 E05

## 案例背景
信用卡欺诈检测是典型的**极度类别不均衡**二分类：正类（欺诈）往往 <1%。业务上漏掉一笔欺诈的代价远高于误杀正常交易，但模型的「准确率」在这种分布下几乎失去区分力——全预测为 0 也能得到 99%+ Accuracy。

## 你将学到
1. **评估指标**：PR-AUC（Average Precision）为何比 ROC-AUC 更贴欺诈场景；Accuracy 陷阱的可视化
2. **验证策略**：按 `Time` 排序切分，禁止随机打乱（避免用「未来」训练过去）
3. **不平衡处理**：`class_weight='balanced'` 作为可上线基线；**SMOTE 仅应在训练折内**使用的故事线
4. **wanglaoshi**：`ready` → `DataAnalyzer` → `QuickBaseline` → `SubmissionGenerator`（`prediction_type='label'`，禁止 `'probability'`）

## 理论速览：PR 曲线 vs ROC 曲线
- **ROC**：横轴 FPR、纵轴 TPR；当负类极多时，FPR 仍可能很低，曲线「看起来很好」
- **PR**：横轴 Recall、纵轴 Precision；正类稀少时，Precision 对阈值极敏感，更反映排序质量
- 本 Notebook **主指标：PR-AUC**；ROC-AUC 仅作对照，不作为选型依据

## 数据
- 本地：`data/creditcard.csv`（Kaggle Credit Card Fraud）
- 缺失：合成 DEMO 不均衡数据（流程等价，结论一致）


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 0. 环境确认


In [ ]:
import pandas as pd
import numpy as np
print('pandas', pd.__version__, 'numpy', np.__version__)
print('QUICK =', QUICK, 'RANDOM_STATE =', RANDOM_STATE)


## 1. 加载真实数据或合成 DEMO


In [ ]:
from pathlib import Path
from sklearn.datasets import make_classification

path = Path('../../data/creditcard.csv')
DEMO = False
if path.exists():
    df = pd.read_csv(path)
    if QUICK and len(df) > 80000:
        pos = df[df['Class'] == 1]
        neg = df[df['Class'] == 0].sample(
            min(20000, (df['Class'] == 0).sum()), random_state=RANDOM_STATE
        )
        df = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_STATE)
    print('loaded', path.resolve(), df.shape)
    print(df['Class'].value_counts(normalize=True))
else:
    DEMO = True
    print('未找到 creditcard.csv，使用合成不均衡 DEMO 数据')
    X_syn, y_syn = make_classification(
        n_samples=8000 if QUICK else 30000,
        n_features=20,
        n_informative=8,
        weights=[0.985, 0.015],
        flip_y=0.01,
        random_state=RANDOM_STATE,
    )
    df = pd.DataFrame(X_syn, columns=[f'V{i}' for i in range(X_syn.shape[1])])
    df['Time'] = np.arange(len(df))
    df['Amount'] = np.abs(np.random.RandomState(RANDOM_STATE).normal(50, 30, len(df)))
    df['Class'] = y_syn
    print(df['Class'].value_counts(normalize=True))
df.head(3)


## 2. EDA：类别比例、金额分布与「准确率幻觉」

极端不均衡时，应先回答：**随机猜全为负，指标是多少？** 这给出任何模型的下限。


In [ ]:
from wanglaoshi.analyze import DataAnalyzer

eda_df = df.sample(min(len(df), 5000), random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
display(analyzer.get_basic_info())
display(analyzer.basic_statistics().head(12))
fraud_rate = float(df['Class'].mean())
print('fraud rate =', fraud_rate)
print('dummy all-zero accuracy ≈', 1 - fraud_rate)
if 'Amount' in df.columns:
    print(df.groupby('Class')['Amount'].agg(['mean', 'median', 'max']))


### 2.1 理论：为何 Accuracy 误导

设欺诈率 \(p pprox 0.002\)。分类器永远输出「正常」可得 Accuracy ≈ 99.8%，但 Recall=0，业务价值为零。因此本案例：
- **主看**：PR-AUC、`classification_report` 中的 precision/recall
- **辅看**：ROC-AUC（了解排序，但不作为唯一依据）


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score

feature_cols = [c for c in df.columns if c not in ('Class', 'Time')]
work = df.sort_values('Time').reset_index(drop=True)
split = int(len(work) * 0.8)
train, valid = work.iloc[:split], work.iloc[split:]
Xtr, Xva = train[feature_cols], valid[feature_cols]
ytr, yva = train['Class'], valid['Class']

dummy = DummyClassifier(strategy='constant', constant=0)
dummy.fit(Xtr, ytr)
pred_d = dummy.predict(Xva)
print('Dummy Accuracy', accuracy_score(yva, pred_d))
print('Dummy PR-AUC   ', average_precision_score(yva, pred_d))


## 3. 时间切分（禁止随机打乱）

欺诈数据按交易时间有序。随机 `train_test_split` 会让训练集混入验证时段之后的信息，离线指标虚高。  
**正确做法**：按 `Time` 排序，前 80% 训练、后 20% 验证（模拟「用历史预测未来」）。


In [ ]:
print('train fraud rate', ytr.mean(), '| valid fraud rate', yva.mean())
print('time train', train['Time'].min(), '->', train['Time'].max())
print('time valid', valid['Time'].min(), '->', valid['Time'].max())
print('n_train', len(train), 'n_valid', len(valid))


## 4. 特征缩放（PCA 匿名特征 + Amount）


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xtr_s = pd.DataFrame(scaler.fit_transform(Xtr), columns=Xtr.columns, index=Xtr.index)
Xva_s = pd.DataFrame(scaler.transform(Xva), columns=Xva.columns, index=Xva.index)
Xtr_s.head(2)


## 5. 基线：`class_weight='balanced'` 与 PR-AUC 选型

`class_weight` 在损失函数里加大正类权重，**不合成新样本**，适合作为生产默认可解释基线。  
`QuickBaseline` 跑多模型时，我们**用手工 PR-AUC 表**重排，而不是默认 Accuracy 排序。


In [ ]:
from wanglaoshi import compete as CT
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_s, ytr, Xva_s, yva, task_type='classification')
comparison = baseline.compare_models(results, top_n=8)
display(comparison)

rows = []
for name, pack in results.items():
    model = pack['model']
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(Xva_s)[:, 1]
    else:
        proba = np.asarray(pack['predictions'], dtype=float)
    rows.append({
        'Model': name,
        'PR-AUC': average_precision_score(yva, proba),
        'ROC-AUC': roc_auc_score(yva, proba),
    })
metric_df = pd.DataFrame(rows).sort_values('PR-AUC', ascending=False).reset_index(drop=True)
print('=== 按 PR-AUC 排序（主指标）===')
display(metric_df)
best = metric_df.iloc[0]['Model']
model = results[best]['model']
if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(Xva_s)[:, 1]
else:
    proba = np.asarray(results[best]['predictions'], dtype=float)
pred = (proba >= 0.5).astype(int)
print(classification_report(yva, pred, zero_division=0))
print('选中模型', best, 'PR-AUC', average_precision_score(yva, proba))


## 6. 显式对比：`class_weight` 与默认权重

下面只换 `LogisticRegression` 的 `class_weight`，观察 **PR-AUC / Recall** 变化（Accuracy 可能下降，这是预期现象）。


In [ ]:
from sklearn.pipeline import Pipeline

for cw in [None, 'balanced']:
    pipe = Pipeline([
        ('clf', LogisticRegression(max_iter=800, random_state=RANDOM_STATE, class_weight=cw)),
    ])
    pipe.fit(Xtr_s, ytr)
    p = pipe.predict_proba(Xva_s)[:, 1]
    pred_pos = (p >= 0.5)
    fraud_recall = ((yva.to_numpy() == 1) & pred_pos).sum() / max(1, (yva == 1).sum())
    print('class_weight=', cw,
          'PR-AUC', round(average_precision_score(yva, p), 4),
          'ROC-AUC', round(roc_auc_score(yva, p), 4),
          'fraud recall@0.5', round(fraud_recall, 4))


## 7. SMOTE 叙事：只在训练折内重采样

**错误**：对整表 SMOTE 后再划分 → 验证集与训练集通过合成样本「泄漏」相似点。  
**正确**：在每个 CV 折里，仅对 **训练折** `fit_resample`，验证折保持原始分布。

本段在**时间有序验证集不变**的前提下，用 `StratifiedKFold` 在训练部分演示折内 SMOTE 的 PR-AUC（教学用途；上线仍以 class_weight + 阈值为主）。


In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

n_splits = 3 if QUICK else 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(Xtr_s))
clf = LogisticRegression(max_iter=600, random_state=RANDOM_STATE, class_weight='balanced')

for fold, (tr_idx, va_idx) in enumerate(cv.split(Xtr_s, ytr)):
    X_f_tr = Xtr_s.iloc[tr_idx]
    y_f_tr = ytr.iloc[tr_idx]
    X_f_va = Xtr_s.iloc[va_idx]
    smote = SMOTE(random_state=RANDOM_STATE + fold, k_neighbors=3)
    try:
        X_res, y_res = smote.fit_resample(X_f_tr, y_f_tr)
    except ValueError:
        X_res, y_res = X_f_tr, y_f_tr
        print(f'fold {fold}: SMOTE skip（正类过少）')
    clf.fit(X_res, y_res)
    oof_proba[va_idx] = clf.predict_proba(X_f_va)[:, 1]

print('OOF PR-AUC (train, fold-safe SMOTE)', average_precision_score(ytr, oof_proba))
print('Hold-out valid PR-AUC (no SMOTE on valid)', average_precision_score(yva, proba))


## 7.1 可视化：PR 曲线 vs ROC 曲线（同一模型）

在同一张图上对比两种曲线，强化「Accuracy/ROC 好看但 PR 反映排序」的直觉。
- PR 曲线在正类稀少时通常**更陡、更敏感**
- 选型以 **Average Precision（PR-AUC）** 为准


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score, roc_auc_score

fpr, tpr, _ = roc_curve(yva, proba)
prec_c, rec_c, _ = precision_recall_curve(yva, proba)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].plot(rec_c, prec_c, label=f'AP={average_precision_score(yva, proba):.3f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('PR curve (主)')
axes[0].axhline(yva.mean(), color='gray', ls='--', label='base rate')
axes[0].legend()
axes[1].plot(fpr, tpr, label=f'AUC={roc_auc_score(yva, proba):.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('ROC curve (辅)')
axes[1].legend()
plt.tight_layout()
plt.show()


## 7.2 金额特征：log1p(Amount) 与欺诈

`Amount` 长尾分布常见。取对数后缩放，有时能提升 PR-AUC（非保证，需实验验证）。


In [ ]:
if 'Amount' in train.columns:
    amt_tr = np.log1p(train['Amount'].astype(float))
    amt_va = np.log1p(valid['Amount'].astype(float))
    Xtr_a = Xtr_s.copy()
    Xva_a = Xva_s.copy()
    Xtr_a['Amount_log'] = StandardScaler().fit_transform(amt_tr.values.reshape(-1, 1))
    Xva_a['Amount_log'] = StandardScaler().fit(amt_tr.values.reshape(-1, 1)).transform(
        amt_va.values.reshape(-1, 1)
    )
    lr = LogisticRegression(max_iter=600, random_state=RANDOM_STATE, class_weight='balanced')
    lr.fit(Xtr_a, ytr)
    p_amt = lr.predict_proba(Xva_a)[:, 1]
    print('PR-AUC with Amount_log', average_precision_score(yva, p_amt))
else:
    print('无 Amount 列，跳过金额工程')


### 7.3 理论补课：不均衡学习菜单

| 方法 | 作用位置 | 上线友好度 | 本 Notebook |
|------|----------|------------|-------------|
| `class_weight` | 损失函数 | 高 | ✅ 基线 |
| 阈值调整 | 推断 | 高 | ✅ 第 8 节 |
| SMOTE | 仅训练折 | 中（需严格 CV） | ✅ 第 7 节 |
| 欠采样 | 训练 | 低（丢信息） | 未展开 |
| 异常检测视角 | 特征空间 | 视场景 | 拓展阅读 |

欺诈系统常见组合：**时间验证 + class_weight + PR-AUC 选型 + 业务阈值**。


## 8. 阈值：固定 Precision 下最大化 Recall


In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, ths = precision_recall_curve(yva, proba)
target_prec = 0.5
mask = prec[:-1] >= target_prec
if mask.any():
    idx = np.argmax(rec[:-1][mask])
    thr = ths[mask][idx]
    print('chosen thr', thr, 'precision', prec[:-1][mask][idx], 'recall', rec[:-1][mask][idx])
else:
    print('无 precision>=0.5 的阈值，请放宽约束或改盯 PR-AUC')


## 9. 模拟提交（标签，非概率）


In [ ]:
from pathlib import Path
from wanglaoshi import compete as CT

out = Path('outputs/05_credit_card_fraud')
out.mkdir(parents=True, exist_ok=True)
pred_label = (proba >= 0.5).astype(int)
sample = pd.DataFrame({'Id': np.arange(len(pred_label)), 'Class': yva.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_label,
    str(out / 'sample_submission.csv'),
    str(out / 'submission.csv'),
    prediction_type='label',
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
print('DEMO mode =', DEMO)
sub.head(3)


## 10. 小结与检查清单
- [x] 时间切分而非随机划分
- [x] **主指标 PR-AUC**，对照 ROC-AUC / Dummy 下限
- [x] `class_weight='balanced'` 基线
- [x] SMOTE 仅在 CV 训练折内（叙事 + 代码）
- [x] 提交 `prediction_type='label'`
- [ ] 上线：监控 precision@流量、人工复核队列
- [ ] 成本敏感阈值（漏报 vs 误报货币化）


## 附录 A：欺诈检测实验设计备忘

**离线实验叙事**：先固定时间切分，再在同一验证集上比较特征、模型、重采样。任何改动若重新划分时间窗，必须与历史实验标注版本号，否则 PR-AUC 不可比。

**为何不用 Accuracy 选型**：正类占比 \(p\) 小时，Accuracy 的 Hessian 对正类极不敏感；Optimizer 会滑向「全负类」鞍点。报告 Accuracy 可作为业务沟通，但不宜写入模型 gates。

**ROC vs PR 再强调**：ROC 积分对负类采样不敏感；PR 积分直接惩罚「把正类排在后面」的行为。Kaggle 类似赛题常见 leaderboard 用 AUC，但银行风控内部更常看 KS、PR-AUC、Lorenz 曲线。

**class_weight 机制**：相当于对正负样本的损失乘系数，不改变训练集行数。优点是流程与生产一致；缺点是若 \(p\) 极小，权重比值大，训练方差升高，应配合早停与正则。

**SMOTE 折内故事**：合成样本只在训练折生成，验证折永远是原始分布，这是唯一 honest 的 PR 曲线。切勿 `fit_resample` 在全训练集后再切验证——那会让验证点落在合成邻域里。

**wanglaoshi API 路径**：`ready` 统一环境 → `DataAnalyzer` 看 fraud rate → `QuickBaseline` 多模型初筛（记得按 PR-AUC 重排）→ `SubmissionGenerator` 输出离散 `Class` 标签。禁止 `prediction_type='probability'` 伪造概率提交。

**Amount / Time**：金额对数化、时间周期特征（小时/星期）是常见增益点；PCA 分量 V1–V28 已不可逆，解释性弱，可用 SHAP 在验证折上抽查。


## 附录 B：阈值与运营成本（定性）

设：漏掉一笔欺诈损失 \(L_{fn}\)，误杀一笔正常交易成本 \(L_{fp}\)。  
最优点满足 \( rac{p(	ext{欺诈}|s)}{p(	ext{正常}|s)} \) 与 \(rac{L_{fp}}{L_{fn}}\) 的权衡。  
当 \(L_{fn} \gg L_{fp}\) 时，应降低阈值、抬高召回，即使 Precision 下降。  
本 Notebook 用「Precision ≥ 0.5 下最大 Recall」只是演示接口，上线阈值须与风控委员会签字版本一致。


## 附录 C：交叉验证 PR-AUC 代码模板（时间序 hold-out 已固定）


In [ ]:
from sklearn.model_selection import cross_val_score

clf_cv = LogisticRegression(max_iter=500, class_weight='balanced', random_state=RANDOM_STATE)
pr_scores = cross_val_score(
    clf_cv, Xtr_s, ytr, cv=3 if QUICK else 5,
    scoring='average_precision', n_jobs=-1,
)
print('CV PR-AUC mean±std', pr_scores.mean(), pr_scores.std())


## 教师导读：欺诈检测课堂节奏

**先打脸 Accuracy**：用 Dummy 全负类，让学生背出 fraud rate 与 dummy accuracy。  
**再上 PR-AUC**：同一验证集，比较模型表与 `average_precision_score` 排序差异。  
**时间切分**：画 train/valid 时间区间，禁止 `shuffle=True` 的 hold-out。  
**class_weight**：对照 None vs balanced 的 fraud recall。  
**SMOTE**：只展示折内代码，用一句话封住「全表 SMOTE 再划分」错误。  
**提交**：只演示 label；刻意不提 probability 类型。

**与 sklearn 文档差异**：我们强调 PR 主 ROC 辅，与 credit card 赛题社区习惯一致。  
**DEMO 数据**：无 CSV 时流程不变，_metrics 数值仅供演示 pipeline。


## 附录 B：阈值与业务约束

即使排序很好（高 AP），上线仍要选阈值满足：
- 日均误报量上限
- 最低召回（监管/风控）

在验证折上画 Precision-Recall 曲线，选工作点；测试折只评估一次。


In [ ]:
print('Practice: precision_recall_curve(y_valid, p_hat) → choose threshold under cost constraints.')


## 深度阅读与作业（Fraud）

### 必读观点（Fraud Detection Handbook）
1. **PR-AUC / AP** 才是主排序指标；Accuracy、甚至 ROC-AUC 都可能误导。
2. 重采样（SMOTE）必须落在训练折内；不少实验显示 `class_weight` 不输 SMOTE。
3. 阈值选择是业务问题：日误报预算 × 最低召回。

### 书面作业
1. 计算「全预测负类」的 Accuracy，并解释为何无用。
2. 比较 ROC-AUC 与 AP 在本数据（或 DEMO）上的数值差距。
3. 画出若先 SMOTE 再划分，泄漏路径的示意图（文字版即可）。


## 课堂讨论记录区（请学生填写）

主题：05_credit_card_fraud

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：05_credit_card_fraud

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：05_credit_card_fraud

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。
